# Step 4C — Oriented BEV Hierarchical Retrieval

This notebook validates the full chain:

**nuScenes ego pose → oriented BEV polygon → fixed z24 cover vs hierarchical BST cover → HD-map retrieval**

It compares:

1. A fixed-resolution polygon cover at the target zoom.
2. The mixed-resolution hierarchical polygon cover from Step 4B.
3. Candidate-set equivalence.
4. Precision/recall against polygon-based oriented ground truth.
5. Lookup latency and total query latency.

The main success criterion is:

> **The hierarchical query returns exactly the same HD-map candidates as the fixed-resolution polygon query, but with far fewer spatial descriptors.**

This notebook assumes these source files already exist and work:

- `src/bst_bev/query/oriented_region.py`
- `src/bst_bev/query/polygon_hierarchical.py`
- `src/bst_bev/nuscenes/pose.py`
- `src/bst_bev/maps/hierarchical_bst_map_index.py`


In [43]:
from pathlib import Path
import sys
import math
import time

import numpy as np
import pandas as pd

from shapely.geometry import box
from shapely.ops import unary_union

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATAROOT = PROJECT_ROOT / "data" / "nuscenes"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Project root:", PROJECT_ROOT)
print("nuScenes root:", DATAROOT)


Project root: /Users/abinavbala/College/Research/BST-BEV
nuScenes root: /Users/abinavbala/College/Research/BST-BEV/data/nuscenes


In [44]:
from nuscenes.nuscenes import NuScenes
from nuscenes.map_expansion.map_api import NuScenesMap

from bst_id.region_algebra import Cell

from bst_bev.nuscenes.pose import quaternion_to_yaw

from bst_bev.query.oriented_region import (
    OrientedBEVRegion,
)

from bst_bev.query.polygon_hierarchical_fast import (
    FastHierarchicalPolygonRegionCover,
)

from bst_bev.maps.hierarchical_bst_map_index import (
    HierarchicalBSTMapIndex,
)


## Configuration

Keep the same **60 m × 60 m** BEV footprint as the earlier Step 3D experiment so the only major change is orientation.

`MIN_ZOOM = 19` matches the optimized hierarchy range you used after Step 3D.


In [45]:
MAP_LAYERS = (
    "lane_divider",
    "road_divider",
    "ped_crossing",
)

BEV_FRONT_M = 30.0
BEV_BACK_M = 30.0
BEV_LEFT_M = 30.0
BEV_RIGHT_M = 30.0

MIN_ZOOM = 19
TARGET_ZOOM = 24

INDEX_SPACING_M = 1.0
BOUNDARY_STEP_M = 1.0

TIMING_REPEATS = 20

# Change this to inspect another nuScenes sample.
SAMPLE_INDEX = 0

RAW_OUTPUT = (
    OUTPUT_DIR
    / "step4c_oriented_retrieval_single_sample.csv"
)


## Load nuScenes and sample metadata


In [46]:
nusc = NuScenes(
    version="v1.0-mini",
    dataroot=str(DATAROOT),
    verbose=False,
)

print("Scenes:", len(nusc.scene))
print("Samples:", len(nusc.sample))


Scenes: 10
Samples: 404


In [47]:
def get_sample_context(
    nusc,
    sample,
):
    scene = nusc.get(
        "scene",
        sample["scene_token"],
    )

    log = nusc.get(
        "log",
        scene["log_token"],
    )

    location = log["location"]

    lidar = nusc.get(
        "sample_data",
        sample["data"]["LIDAR_TOP"],
    )

    ego_pose = nusc.get(
        "ego_pose",
        lidar["ego_pose_token"],
    )

    ego_x, ego_y, _ = (
        ego_pose["translation"]
    )

    ego_yaw = quaternion_to_yaw(
        ego_pose["rotation"]
    )

    return {
        "sample_token":
            sample["token"],

        "scene":
            scene["name"],

        "location":
            location,

        "ego_x":
            float(ego_x),

        "ego_y":
            float(ego_y),

        "ego_yaw":
            float(ego_yaw),
    }


samples_df = pd.DataFrame(
    get_sample_context(
        nusc,
        sample,
    )
    for sample in nusc.sample
)

print("Sample rows:", len(samples_df))
samples_df.head()


Sample rows: 404


,sample_token,scene,location,ego_x,ego_y,ego_yaw
0,ca9a282c9e77460f8360f564131a8af5,scene-0061,singapore-onenorth,411.303935,1180.890379,-1.923645
1,39586f9d59004284a7114a68825e8eec,scene-0061,singapore-onenorth,409.743152,1176.676973,-1.917831
2,356d81f38dd9473ba590f39e266f54e5,scene-0061,singapore-onenorth,408.101879,1172.151578,-1.911575
3,e0845f5322254dafadbbed75aaa07969,scene-0061,singapore-onenorth,406.726006,1168.211287,-1.900030
4,c923fe08b2ff4e27975d2bf30934383b,scene-0061,singapore-onenorth,405.409406,1164.354227,-1.895311


In [48]:
locations = sorted(
    samples_df["location"].unique()
)

nusc_maps = {
    location: NuScenesMap(
        dataroot=str(DATAROOT),
        map_name=location,
    )
    for location in locations
}

print("Loaded maps:")
for location in locations:
    print(" -", location)


Loaded maps:
 - boston-seaport
 - singapore-hollandvillage
 - singapore-onenorth
 - singapore-queenstown


## Select one test sample and build the oriented BEV


In [49]:
sample = samples_df.iloc[
    SAMPLE_INDEX
]

location = str(
    sample["location"]
)

bev = OrientedBEVRegion(
    center_x=float(
        sample["ego_x"]
    ),

    center_y=float(
        sample["ego_y"]
    ),

    yaw=float(
        sample["ego_yaw"]
    ),

    front=BEV_FRONT_M,
    back=BEV_BACK_M,

    left=BEV_LEFT_M,
    right=BEV_RIGHT_M,
)

print("Sample index:", SAMPLE_INDEX)
print("Scene:", sample["scene"])
print("Location:", location)
print(
    "Ego:",
    (
        bev.center_x,
        bev.center_y,
    ),
)
print(
    "Yaw [deg]:",
    math.degrees(bev.yaw),
)
print(
    "BEV area [m^2]:",
    bev.area,
)
print(
    "Polygon valid:",
    bev.polygon.is_valid,
)


Sample index: 0
Scene: scene-0061
Location: singapore-onenorth
Ego: (411.3039349319818, 1180.8903791765097)
Yaw [deg]: -110.21676287299779
BEV area [m^2]: 3600.0
Polygon valid: True


## Polygon-based oriented ground truth

This is the retrieval ground truth for Step 4C. It intersects the actual map geometry with the rotated BEV polygon.


In [50]:
def get_map_record_geometry(
    nusc_map,
    layer,
    record,
):
    if layer in (
        "lane_divider",
        "road_divider",
    ):
        return nusc_map.extract_line(
            record["line_token"]
        )

    if layer == "ped_crossing":
        return nusc_map.extract_polygon(
            record["polygon_token"]
        )

    raise ValueError(
        f"Unsupported layer: {layer}"
    )


def get_polygon_ground_truth_tokens(
    nusc_map,
    query_polygon,
    map_layers=MAP_LAYERS,
):
    hits = set()

    for layer in map_layers:
        records = getattr(
            nusc_map,
            layer,
        )

        for record in records:
            geometry = (
                get_map_record_geometry(
                    nusc_map,
                    layer,
                    record,
                )
            )

            if geometry.intersects(
                query_polygon
            ):
                hits.add(
                    (
                        layer,
                        record["token"],
                    )
                )

    return hits


In [51]:
test_map = nusc_maps[
    location
]

oriented_gt = (
    get_polygon_ground_truth_tokens(
        nusc_map=test_map,
        query_polygon=bev.polygon,
    )
)

print(
    "Oriented ground-truth objects:",
    len(oriented_gt),
)

sorted(oriented_gt)[:20]


Oriented ground-truth objects: 4


[('lane_divider', '3d37fc34-730c-4894-9d8a-6ddbcd2aacbd'),
 ('lane_divider', '425d4a61-7336-4eab-ad58-7615f79d63bb'),
 ('lane_divider', '6b5a4771-79b4-4ddf-b305-5f4f8260b6ac'),
 ('lane_divider', '80fedd56-10ee-4109-8519-56c904230158')]

## Build the hierarchical HD-map index

This is an offline cost. For the real all-sample experiment, build it **once per map location** and reuse it for all frames.


In [52]:
hierarchical_index = (
    HierarchicalBSTMapIndex(
        nusc_map=test_map,
        location=location,
        min_zoom=MIN_ZOOM,
        max_zoom=TARGET_ZOOM,
        index_spacing_m=INDEX_SPACING_M,
    )
)

build_start = time.perf_counter()

hierarchical_index.build()

build_time_s = (
    time.perf_counter()
    - build_start
)

print(
    "Hierarchical index keys:",
    f"{hierarchical_index.key_count:,}",
)

print(
    "Primitive memberships:",
    f"{hierarchical_index.membership_count:,}",
)

print(
    "Build time:",
    f"{build_time_s:.3f} s",
)


Hierarchical index keys: 103,158
Primitive memberships: 148,255
Build time: 1.157 s


## Build the Step 4B hierarchical polygon cover


In [53]:
polygon_coverer = (
    FastHierarchicalPolygonRegionCover(
        encoder=hierarchical_index.encoder,
        min_zoom=MIN_ZOOM,
        target_zoom=TARGET_ZOOM,
        boundary_step_m=BOUNDARY_STEP_M,
    )
)

hierarchical_cover = (
    polygon_coverer.cover(
        bev
    )
)

print(
    "Hierarchical cells:",
    hierarchical_cover.hierarchical_cell_count,
)

print(
    "Leaf-equivalent count:",
    hierarchical_cover.leaf_equivalent_count,
)

print(
    "Cover time:",
    f"{hierarchical_cover.cover_time_ms:.4f} ms",
)


Hierarchical cells: 206
Leaf-equivalent count: 698
Cover time: 7.7181 ms


## Conservative-cover sanity check

The hierarchical BST cell union must fully cover the projected oriented BEV polygon.


In [54]:
hierarchical_cell_polygons = []

for cell in hierarchical_cover.cells:
    bounds = (
        polygon_coverer.cell_bounds(
            cell
        )
    )

    hierarchical_cell_polygons.append(
        box(
            *bounds
        )
    )

hierarchical_union = unary_union(
    hierarchical_cell_polygons
)

uncovered = (
    hierarchical_cover.tile_polygon
    .difference(
        hierarchical_union
    )
)

polygon_area = (
    hierarchical_cover.tile_polygon.area
)

hierarchical_cover_area = (
    hierarchical_union.area
)

overcoverage_ratio = (
    (
        hierarchical_cover_area
        - polygon_area
    )
    / polygon_area
)

print(
    "Cover contains BEV:",
    hierarchical_union.covers(
        hierarchical_cover.tile_polygon
    ),
)

print(
    "Uncovered area:",
    uncovered.area,
)

print(
    "Over-coverage:",
    f"{100 * overcoverage_ratio:.4f}%",
)


Cover contains BEV: True
Uncovered area: 0.0
Over-coverage: 10.5700%


## Fixed-resolution polygon cover

This is the Step 4C baseline: every target-zoom BST leaf cell that intersects the same oriented polygon.


In [55]:
def fixed_leaf_polygon_cover(
    tile_polygon,
    target_zoom,
):
    min_x, min_y, max_x, max_y = (
        tile_polygon.bounds
    )

    x_start = math.floor(
        min_x
    )

    x_end = math.floor(
        max_x
    )

    y_start = math.floor(
        min_y
    )

    y_end = math.floor(
        max_y
    )

    cells = []

    for tile_x in range(
        x_start,
        x_end + 1,
    ):
        for tile_y in range(
            y_start,
            y_end + 1,
        ):
            leaf_polygon = box(
                tile_x,
                tile_y,
                tile_x + 1,
                tile_y + 1,
            )

            if leaf_polygon.disjoint(
                tile_polygon
            ):
                continue

            cells.append(
                Cell.from_parts(
                    x=(
                        target_zoom,
                        tile_x,
                    ),

                    y=(
                        target_zoom,
                        tile_y,
                    ),
                )
            )

    return tuple(
        cells
    )


fixed_cells = (
    fixed_leaf_polygon_cover(
        tile_polygon=(
            hierarchical_cover.tile_polygon
        ),

        target_zoom=(
            TARGET_ZOOM
        ),
    )
)

print(
    "Fixed z24 cells:",
    len(fixed_cells),
)

print(
    "Hierarchical cells:",
    hierarchical_cover.hierarchical_cell_count,
)

print(
    "Descriptor compression:",
    f"{len(fixed_cells) / hierarchical_cover.hierarchical_cell_count:.2f}x",
)


Fixed z24 cells: 698
Hierarchical cells: 206
Descriptor compression: 3.39x


## Query the HD-map index both ways

Both queries use the **same hierarchical index**. This isolates the query-representation difference:

- fixed z24 leaf descriptors
- compact mixed-resolution descriptors


In [56]:
fixed_result = (
    hierarchical_index.query_cells(
        fixed_cells
    )
)

fixed_tokens = (
    fixed_result.tokens
)


hierarchical_result = (
    hierarchical_index.query_cells(
        hierarchical_cover.cells
    )
)

hierarchical_tokens = (
    hierarchical_result.tokens
)


candidate_union = (
    fixed_tokens
    | hierarchical_tokens
)

candidate_intersection = (
    fixed_tokens
    & hierarchical_tokens
)

candidate_jaccard = (
    len(candidate_intersection)
    / len(candidate_union)
    if candidate_union
    else 1.0
)

same_candidates = (
    fixed_tokens
    == hierarchical_tokens
)

print(
    "Fixed candidates:",
    len(fixed_tokens),
)

print(
    "Hierarchical candidates:",
    len(hierarchical_tokens),
)

print(
    "Same candidates:",
    same_candidates,
)

print(
    "Candidate Jaccard:",
    candidate_jaccard,
)

print(
    "Fixed only:",
    fixed_tokens
    - hierarchical_tokens,
)

print(
    "Hierarchical only:",
    hierarchical_tokens
    - fixed_tokens,
)


Fixed candidates: 4
Hierarchical candidates: 4
Same candidates: True
Candidate Jaccard: 1.0
Fixed only: set()
Hierarchical only: set()


## Retrieval metrics against oriented ground truth


In [57]:
def retrieval_metrics(
    candidates,
    ground_truth,
):
    tp = len(
        candidates
        & ground_truth
    )

    fp = len(
        candidates
        - ground_truth
    )

    fn = len(
        ground_truth
        - candidates
    )

    precision = (
        tp / (tp + fp)
        if tp + fp
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn
        else 0.0
    )

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
    }


fixed_metrics = retrieval_metrics(
    fixed_tokens,
    oriented_gt,
)

hierarchical_metrics = retrieval_metrics(
    hierarchical_tokens,
    oriented_gt,
)

print(
    "Fixed metrics:",
    fixed_metrics,
)

print(
    "Hierarchical metrics:",
    hierarchical_metrics,
)


Fixed metrics: {'tp': 4, 'fp': 0, 'fn': 0, 'precision': 1.0, 'recall': 1.0}
Hierarchical metrics: {'tp': 4, 'fp': 0, 'fn': 0, 'precision': 1.0, 'recall': 1.0}


## Stable timing helpers

The operations are short, so this notebook uses repeated timings and reports medians.


In [58]:
def median_runtime_ms(
    fn,
    repeats=TIMING_REPEATS,
):
    timings = []

    for _ in range(
        repeats
    ):
        start = (
            time.perf_counter()
        )

        fn()

        timings.append(
            (
                time.perf_counter()
                - start
            )
            * 1000.0
        )

    return float(
        np.median(
            timings
        )
    )


## Benchmark fixed and hierarchical cover generation


In [59]:
fixed_cover_time_ms = (
    median_runtime_ms(
        lambda:
        fixed_leaf_polygon_cover(
            hierarchical_cover.tile_polygon,
            TARGET_ZOOM,
        )
    )
)

hierarchical_cover_time_ms = (
    median_runtime_ms(
        lambda:
        polygon_coverer.cover(
            bev
        )
    )
)

print(
    "Fixed cover generation:",
    f"{fixed_cover_time_ms:.4f} ms",
)

print(
    "Hierarchical cover generation:",
    f"{hierarchical_cover_time_ms:.4f} ms",
)


Fixed cover generation: 29.3220 ms
Hierarchical cover generation: 7.6426 ms


## Benchmark index lookup


In [60]:
fixed_lookup_time_ms = (
    median_runtime_ms(
        lambda:
        hierarchical_index.query_cells(
            fixed_cells
        )
    )
)

hierarchical_lookup_time_ms = (
    median_runtime_ms(
        lambda:
        hierarchical_index.query_cells(
            hierarchical_cover.cells
        )
    )
)

print(
    "Fixed lookup:",
    f"{fixed_lookup_time_ms:.4f} ms",
)

print(
    "Hierarchical lookup:",
    f"{hierarchical_lookup_time_ms:.4f} ms",
)

print(
    "Lookup speedup:",
    f"{fixed_lookup_time_ms / hierarchical_lookup_time_ms:.2f}x",
)


Fixed lookup: 2.0844 ms
Hierarchical lookup: 0.5855 ms
Lookup speedup: 3.56x


## Total online latency

For a fair online comparison:

**total = cover generation + index lookup**


In [61]:
fixed_total_time_ms = (
    fixed_cover_time_ms
    + fixed_lookup_time_ms
)

hierarchical_total_time_ms = (
    hierarchical_cover_time_ms
    + hierarchical_lookup_time_ms
)

total_speedup = (
    fixed_total_time_ms
    / hierarchical_total_time_ms
)

print(
    "Fixed total:",
    f"{fixed_total_time_ms:.4f} ms",
)

print(
    "Hierarchical total:",
    f"{hierarchical_total_time_ms:.4f} ms",
)

print(
    "Total speedup:",
    f"{total_speedup:.2f}x",
)


Fixed total: 31.4064 ms
Hierarchical total: 8.2281 ms
Total speedup: 3.82x


## Final Step 4C result


In [62]:
result = {
    "sample_index":
        SAMPLE_INDEX,

    "sample_token":
        sample["sample_token"],

    "scene":
        sample["scene"],

    "location":
        location,

    "ego_x":
        bev.center_x,

    "ego_y":
        bev.center_y,

    "ego_yaw_deg":
        math.degrees(
            bev.yaw
        ),

    "target_zoom":
        TARGET_ZOOM,

    "min_zoom":
        MIN_ZOOM,

    # ----------------------------------
    # Query representation
    # ----------------------------------

    "fixed_cells":
        len(fixed_cells),

    "hierarchical_cells":
        hierarchical_cover.hierarchical_cell_count,

    "compression_ratio": (
        len(fixed_cells)
        / hierarchical_cover.hierarchical_cell_count
    ),

    "overcoverage_percent":
        100 * overcoverage_ratio,

    # ----------------------------------
    # Candidates
    # ----------------------------------

    "fixed_candidates":
        len(fixed_tokens),

    "hierarchical_candidates":
        len(hierarchical_tokens),

    "same_candidates":
        same_candidates,

    "candidate_jaccard":
        candidate_jaccard,

    # ----------------------------------
    # Retrieval
    # ----------------------------------

    "ground_truth":
        len(oriented_gt),

    "precision":
        hierarchical_metrics[
            "precision"
        ],

    "recall":
        hierarchical_metrics[
            "recall"
        ],

    "tp":
        hierarchical_metrics[
            "tp"
        ],

    "fp":
        hierarchical_metrics[
            "fp"
        ],

    "fn":
        hierarchical_metrics[
            "fn"
        ],

    # ----------------------------------
    # Timing
    # ----------------------------------

    "fixed_cover_time_ms":
        fixed_cover_time_ms,

    "hierarchical_cover_time_ms":
        hierarchical_cover_time_ms,

    "fixed_lookup_time_ms":
        fixed_lookup_time_ms,

    "hierarchical_lookup_time_ms":
        hierarchical_lookup_time_ms,

    "fixed_total_time_ms":
        fixed_total_time_ms,

    "hierarchical_total_time_ms":
        hierarchical_total_time_ms,

    "lookup_speedup": (
        fixed_lookup_time_ms
        / hierarchical_lookup_time_ms
    ),

    "total_speedup":
        total_speedup,

    # ----------------------------------
    # Index
    # ----------------------------------

    "hierarchical_index_keys":
        hierarchical_index.key_count,

    "hierarchical_memberships":
        hierarchical_index.membership_count,

    "hierarchical_index_build_time_s":
        build_time_s,
}

result_df = pd.DataFrame(
    [result]
)

result_df.T


,0
sample_index,0
sample_token,ca9a282c9e77460f8360f564131a8af5
scene,scene-0061
location,singapore-onenorth
ego_x,411.303935
ego_y,1180.890379
ego_yaw_deg,-110.216763
target_zoom,24
min_zoom,19
fixed_cells,698


In [63]:
result_df.to_csv(
    RAW_OUTPUT,
    index=False,
)

print(
    "Saved:",
    RAW_OUTPUT,
)


Saved: /Users/abinavbala/College/Research/BST-BEV/outputs/step4c_oriented_retrieval_single_sample.csv


## Step 4C success criteria

Step 4C is successful if:

1. `same_candidates == True`
2. `candidate_jaccard == 1.0`
3. Fixed and hierarchical precision/recall are identical.
4. `recall == 1.0` is strongly preferred.
5. The hierarchical query uses substantially fewer descriptors than the fixed z24 polygon cover.
6. The hierarchical cover fully contains the oriented BEV polygon.
7. The total online latency is competitive or faster.

If this single-frame test passes, the next step is **Step 4D: run the same oriented-BEV comparison over all 404 nuScenes-mini samples**.


In [64]:
# ============================================================
# STEP 4D — ALL-SAMPLE ORIENTED BEV EXPERIMENT
# ============================================================

ZOOMS_4D = [
    23,
    24,
]

MIN_ZOOM_4D = 19
MAX_ZOOM_4D = 24

TIMING_REPEATS_4D = 5

STEP4D_RAW_OUTPUT = (
    OUTPUT_DIR
    / "step4d_oriented_all_samples.csv"
)

STEP4D_SUMMARY_OUTPUT = (
    OUTPUT_DIR
    / "step4d_oriented_summary.csv"
)

STEP4D_STORAGE_OUTPUT = (
    OUTPUT_DIR
    / "step4d_oriented_storage.csv"
)

In [65]:
def get_oriented_ground_truth_tokens(
    nusc_map,
    query_polygon,
    map_layers=MAP_LAYERS,
):
    min_x, min_y, max_x, max_y = (
        query_polygon.bounds
    )

    # Cheap coarse candidate retrieval.
    patch_candidates = (
        nusc_map.get_records_in_patch(
            (
                min_x,
                min_y,
                max_x,
                max_y,
            ),
            list(map_layers),
            mode="intersect",
        )
    )

    hits = set()

    for layer, tokens in (
        patch_candidates.items()
    ):
        for token in tokens:

            record = nusc_map.get(
                layer,
                token,
            )

            geometry = (
                get_map_record_geometry(
                    nusc_map,
                    layer,
                    record,
                )
            )

            if geometry.intersects(
                query_polygon
            ):
                hits.add(
                    (
                        layer,
                        token,
                    )
                )

    return hits

In [66]:
oriented_gt_cache = {}

gt_start = time.perf_counter()


for _, row in samples_df.iterrows():

    location = str(
        row["location"]
    )

    bev = OrientedBEVRegion(
        center_x=float(
            row["ego_x"]
        ),

        center_y=float(
            row["ego_y"]
        ),

        yaw=float(
            row["ego_yaw"]
        ),

        front=BEV_FRONT_M,
        back=BEV_BACK_M,

        left=BEV_LEFT_M,
        right=BEV_RIGHT_M,
    )

    oriented_gt_cache[
        row["sample_token"]
    ] = (
        get_oriented_ground_truth_tokens(
            nusc_map=nusc_maps[
                location
            ],
            query_polygon=bev.polygon,
        )
    )


gt_time_s = (
    time.perf_counter()
    - gt_start
)

print(
    "Cached oriented GT for",
    len(oriented_gt_cache),
    "samples",
)

print(
    "GT preprocessing:",
    f"{gt_time_s:.2f}s",
)

Cached oriented GT for 404 samples
GT preprocessing: 5.66s


In [67]:
def median_runtime_ms_4d(
    fn,
):
    timings = []

    for _ in range(
        TIMING_REPEATS_4D
    ):
        start = (
            time.perf_counter()
        )

        fn()

        timings.append(
            (
                time.perf_counter()
                - start
            )
            * 1000.0
        )

    return float(
        np.median(
            timings
        )
    )

In [68]:
def evaluate_oriented_sample(
    *,
    row,
    gt_tokens,
    hierarchical_index,
    hierarchical_coverer,
    target_zoom,
):
    # ========================================================
    # Build BEV
    # ========================================================

    bev = OrientedBEVRegion(
        center_x=float(
            row["ego_x"]
        ),

        center_y=float(
            row["ego_y"]
        ),

        yaw=float(
            row["ego_yaw"]
        ),

        front=BEV_FRONT_M,
        back=BEV_BACK_M,

        left=BEV_LEFT_M,
        right=BEV_RIGHT_M,
    )

    # ========================================================
    # Construct hierarchical cover once
    # ========================================================

    hierarchical_cover = (
        hierarchical_coverer.cover(
            bev
        )
    )

    # ========================================================
    # Fixed target-zoom polygon cover
    # ========================================================

    fixed_cells = (
        fixed_leaf_polygon_cover(
            tile_polygon=(
                hierarchical_cover.tile_polygon
            ),
            target_zoom=target_zoom,
        )
    )

    # ========================================================
    # Retrieve candidates
    # ========================================================

    fixed_result = (
        hierarchical_index.query_cells(
            fixed_cells
        )
    )

    hierarchical_result = (
        hierarchical_index.query_cells(
            hierarchical_cover.cells
        )
    )

    fixed_tokens = (
        fixed_result.tokens
    )

    hierarchical_tokens = (
        hierarchical_result.tokens
    )

    # ========================================================
    # Candidate equivalence
    # ========================================================

    candidate_union = (
        fixed_tokens
        | hierarchical_tokens
    )

    candidate_intersection = (
        fixed_tokens
        & hierarchical_tokens
    )

    candidate_jaccard = (
        len(candidate_intersection)
        / len(candidate_union)
        if candidate_union
        else 1.0
    )

    same_candidates = (
        fixed_tokens
        == hierarchical_tokens
    )

    # ========================================================
    # Accuracy
    # ========================================================

    fixed_metrics = (
        retrieval_metrics(
            fixed_tokens,
            gt_tokens,
        )
    )

    hierarchical_metrics = (
        retrieval_metrics(
            hierarchical_tokens,
            gt_tokens,
        )
    )

    # ========================================================
    # Representation metrics
    # ========================================================

    fixed_cell_count = len(
        fixed_cells
    )

    hierarchical_cell_count = (
        hierarchical_cover
        .hierarchical_cell_count
    )

    descriptor_compression = (
        fixed_cell_count
        / hierarchical_cell_count
        if hierarchical_cell_count
        else np.nan
    )

    descriptor_reduction = (
        1.0
        - hierarchical_cell_count
        / fixed_cell_count
        if fixed_cell_count
        else 0.0
    )

    # ----------------------------------------
    # Coverage area
    #
    # Each target-zoom leaf has unit area in
    # tile space.
    #
    # Hierarchical leaf_equivalent_count is
    # therefore also the exact tile-space
    # area of the non-overlapping BST cover.
    # ----------------------------------------

    polygon_area = (
        hierarchical_cover
        .tile_polygon
        .area
    )

    fixed_cover_area = float(
        fixed_cell_count
    )

    hierarchical_cover_area = float(
        hierarchical_cover
        .leaf_equivalent_count
    )

    fixed_overcoverage_ratio = (
        (
            fixed_cover_area
            - polygon_area
        )
        / polygon_area
    )

    hierarchical_overcoverage_ratio = (
        (
            hierarchical_cover_area
            - polygon_area
        )
        / polygon_area
    )

    # ========================================================
    # Timing: cover construction
    # ========================================================

    # We need one stable projected tile polygon for the
    # fixed baseline.
    tile_polygon = (
        hierarchical_cover
        .tile_polygon
    )

    fixed_cover_time_ms = (
        median_runtime_ms_4d(
            lambda:
            fixed_leaf_polygon_cover(
                tile_polygon=tile_polygon,
                target_zoom=target_zoom,
            )
        )
    )

    hierarchical_cover_time_ms = (
        median_runtime_ms_4d(
            lambda:
            hierarchical_coverer.cover(
                bev
            )
        )
    )

    # ========================================================
    # Timing: index lookup
    # ========================================================

    fixed_lookup_time_ms = (
        median_runtime_ms_4d(
            lambda:
            hierarchical_index.query_cells(
                fixed_cells
            )
        )
    )

    hierarchical_lookup_time_ms = (
        median_runtime_ms_4d(
            lambda:
            hierarchical_index.query_cells(
                hierarchical_cover.cells
            )
        )
    )

    # ========================================================
    # Total online latency
    # ========================================================

    fixed_total_time_ms = (
        fixed_cover_time_ms
        + fixed_lookup_time_ms
    )

    hierarchical_total_time_ms = (
        hierarchical_cover_time_ms
        + hierarchical_lookup_time_ms
    )

    total_speedup = (
        fixed_total_time_ms
        / hierarchical_total_time_ms
        if hierarchical_total_time_ms > 0
        else np.nan
    )

    lookup_speedup = (
        fixed_lookup_time_ms
        / hierarchical_lookup_time_ms
        if hierarchical_lookup_time_ms > 0
        else np.nan
    )

    # ========================================================
    # Return
    # ========================================================

    return {
        # --------------------------------
        # Region representation
        # --------------------------------

        "fixed_cells":
            fixed_cell_count,

        "hierarchical_cells":
            hierarchical_cell_count,

        "compression_ratio":
            descriptor_compression,

        "reduction_fraction":
            descriptor_reduction,

        "polygon_area_tile":
            polygon_area,

        "fixed_overcoverage_ratio":
            fixed_overcoverage_ratio,

        "hierarchical_overcoverage_ratio":
            hierarchical_overcoverage_ratio,

        # --------------------------------
        # Traversal diagnostics
        # --------------------------------

        "visited_cells":
            hierarchical_cover
            .visited_cell_count,

        "interior_cells":
            hierarchical_cover
            .interior_cell_count,

        "boundary_cells":
            hierarchical_cover
            .boundary_cell_count,

        "discarded_cells":
            hierarchical_cover
            .discarded_cell_count,

        # --------------------------------
        # Candidates
        # --------------------------------

        "fixed_candidates":
            len(fixed_tokens),

        "hierarchical_candidates":
            len(hierarchical_tokens),

        "same_candidates":
            same_candidates,

        "candidate_jaccard":
            candidate_jaccard,

        "fixed_only_count":
            len(
                fixed_tokens
                - hierarchical_tokens
            ),

        "hierarchical_only_count":
            len(
                hierarchical_tokens
                - fixed_tokens
            ),

        # --------------------------------
        # Ground truth
        # --------------------------------

        "ground_truth":
            len(gt_tokens),

        # --------------------------------
        # Fixed retrieval
        # --------------------------------

        "fixed_tp":
            fixed_metrics["tp"],

        "fixed_fp":
            fixed_metrics["fp"],

        "fixed_fn":
            fixed_metrics["fn"],

        "fixed_precision":
            fixed_metrics[
                "precision"
            ],

        "fixed_recall":
            fixed_metrics[
                "recall"
            ],

        # --------------------------------
        # Hierarchical retrieval
        # --------------------------------

        "hierarchical_tp":
            hierarchical_metrics["tp"],

        "hierarchical_fp":
            hierarchical_metrics["fp"],

        "hierarchical_fn":
            hierarchical_metrics["fn"],

        "hierarchical_precision":
            hierarchical_metrics[
                "precision"
            ],

        "hierarchical_recall":
            hierarchical_metrics[
                "recall"
            ],

        # --------------------------------
        # Timing
        # --------------------------------

        "fixed_cover_time_ms":
            fixed_cover_time_ms,

        "hierarchical_cover_time_ms":
            hierarchical_cover_time_ms,

        "fixed_lookup_time_ms":
            fixed_lookup_time_ms,

        "hierarchical_lookup_time_ms":
            hierarchical_lookup_time_ms,

        "fixed_total_time_ms":
            fixed_total_time_ms,

        "hierarchical_total_time_ms":
            hierarchical_total_time_ms,

        "lookup_speedup":
            lookup_speedup,

        "total_speedup":
            total_speedup,
    }

In [69]:
import gc


all_4d_results = []
storage_4d_results = []

experiment_start = (
    time.perf_counter()
)


for location in locations:

    print()
    print("=" * 70)
    print(location)
    print("=" * 70)

    nusc_map = (
        nusc_maps[
            location
        ]
    )

    location_samples = (
        samples_df[
            samples_df["location"]
            == location
        ]
    )

    # ========================================================
    # Build one hierarchical map index for this location.
    #
    # It supports every query level from 19 -> 24.
    # ========================================================

    print(
        "Building hierarchical map index..."
    )

    build_start = (
        time.perf_counter()
    )

    hierarchical_index = (
        HierarchicalBSTMapIndex(
            nusc_map=nusc_map,
            location=location,

            min_zoom=(
                MIN_ZOOM_4D
            ),

            max_zoom=(
                MAX_ZOOM_4D
            ),

            index_spacing_m=(
                INDEX_SPACING_M
            ),
        )
    )

    hierarchical_index.build()

    build_time_s = (
        time.perf_counter()
        - build_start
    )

    print(
        "  keys:",
        f"{hierarchical_index.key_count:,}",
    )

    print(
        "  memberships:",
        f"{hierarchical_index.membership_count:,}",
    )

    print(
        "  build:",
        f"{build_time_s:.2f}s",
    )

    storage_4d_results.append({
        "location":
            location,

        "hierarchical_keys":
            hierarchical_index.key_count,

        "hierarchical_memberships":
            hierarchical_index.membership_count,

        "hierarchical_build_time_s":
            build_time_s,
    })

    # ========================================================
    # z23 and z24
    # ========================================================

    for zoom in ZOOMS_4D:

        print()
        print(
            f"  Zoom {zoom}"
        )

        hierarchical_coverer = (
            FastHierarchicalPolygonRegionCover(
                encoder=(
                    hierarchical_index
                    .encoder
                ),

                min_zoom=(
                    MIN_ZOOM_4D
                ),

                target_zoom=zoom,

                boundary_step_m=(
                    BOUNDARY_STEP_M
                ),
            )
        )

        # ----------------------------------------------------
        # Warm-up
        # ----------------------------------------------------

        warmup_row = (
            location_samples.iloc[0]
        )

        warmup_bev = (
            OrientedBEVRegion(
                center_x=float(
                    warmup_row["ego_x"]
                ),

                center_y=float(
                    warmup_row["ego_y"]
                ),

                yaw=float(
                    warmup_row["ego_yaw"]
                ),

                front=BEV_FRONT_M,
                back=BEV_BACK_M,

                left=BEV_LEFT_M,
                right=BEV_RIGHT_M,
            )
        )

        warmup_cover = (
            hierarchical_coverer.cover(
                warmup_bev
            )
        )

        warmup_fixed = (
            fixed_leaf_polygon_cover(
                tile_polygon=(
                    warmup_cover
                    .tile_polygon
                ),

                target_zoom=zoom,
            )
        )

        hierarchical_index.query_cells(
            warmup_cover.cells
        )

        hierarchical_index.query_cells(
            warmup_fixed
        )

        # ----------------------------------------------------
        # All samples in this map
        # ----------------------------------------------------

        for sample_number, (
            _,
            row,
        ) in enumerate(
            location_samples.iterrows(),
            start=1,
        ):

            gt_tokens = (
                oriented_gt_cache[
                    row["sample_token"]
                ]
            )

            metrics = (
                evaluate_oriented_sample(
                    row=row,

                    gt_tokens=(
                        gt_tokens
                    ),

                    hierarchical_index=(
                        hierarchical_index
                    ),

                    hierarchical_coverer=(
                        hierarchical_coverer
                    ),

                    target_zoom=zoom,
                )
            )

            all_4d_results.append({
                "zoom":
                    zoom,

                "scene":
                    row["scene"],

                "sample_token":
                    row["sample_token"],

                "location":
                    location,

                "ego_x":
                    row["ego_x"],

                "ego_y":
                    row["ego_y"],

                "ego_yaw_rad":
                    row["ego_yaw"],

                "ego_yaw_deg":
                    math.degrees(
                        row["ego_yaw"]
                    ),

                **metrics,
            })

            if (
                sample_number % 50
                == 0
            ):
                print(
                    f"    "
                    f"{sample_number}/"
                    f"{len(location_samples)}"
                )

        # ----------------------------------------------------
        # Checkpoint after every location/zoom.
        # ----------------------------------------------------

        results_4d_df = (
            pd.DataFrame(
                all_4d_results
            )
        )

        results_4d_df.to_csv(
            STEP4D_RAW_OUTPUT,
            index=False,
        )

        storage_4d_df = (
            pd.DataFrame(
                storage_4d_results
            )
        )

        storage_4d_df.to_csv(
            STEP4D_STORAGE_OUTPUT,
            index=False,
        )

        zoom_location_rows = (
            results_4d_df[
                (
                    results_4d_df[
                        "zoom"
                    ]
                    == zoom
                )
                &
                (
                    results_4d_df[
                        "location"
                    ]
                    == location
                )
            ]
        )

        print(
            "    samples:",
            len(
                zoom_location_rows
            ),
        )

        print(
            "    agreement:",
            f"{zoom_location_rows['same_candidates'].mean():.3f}",
        )

        print(
            "    median compression:",
            f"{zoom_location_rows['compression_ratio'].median():.2f}x",
        )

        print(
            "    median speedup:",
            f"{zoom_location_rows['total_speedup'].median():.2f}x",
        )

    del hierarchical_index

    gc.collect()


experiment_time_s = (
    time.perf_counter()
    - experiment_start
)

print()
print(
    "Step 4D completed in",
    f"{experiment_time_s / 60:.2f}",
    "minutes",
)

print(
    "Rows:",
    len(
        results_4d_df
    ),
)


boston-seaport
Building hierarchical map index...
  keys: 297,437
  memberships: 466,330
  build: 2.16s

  Zoom 23
    50/163
    100/163
    150/163
    samples: 163
    agreement: 1.000
    median compression: 2.41x
    median speedup: 2.27x

  Zoom 24
    50/163
    100/163
    150/163
    samples: 163
    agreement: 1.000
    median compression: 4.24x
    median speedup: 4.49x

singapore-hollandvillage
Building hierarchical map index...
  keys: 70,414
  memberships: 100,162
  build: 0.36s

  Zoom 23
    50/121
    100/121
    samples: 121
    agreement: 1.000
    median compression: 1.95x
    median speedup: 2.26x

  Zoom 24
    50/121
    100/121
    samples: 121
    agreement: 1.000
    median compression: 3.29x
    median speedup: 4.38x

singapore-onenorth
Building hierarchical map index...
  keys: 103,158
  memberships: 148,255
  build: 0.79s

  Zoom 23
    samples: 39
    agreement: 1.000
    median compression: 2.01x
    median speedup: 2.14x

  Zoom 24
    samples: 39
    a

In [70]:
agreement_4d = (
    results_4d_df
    .groupby("zoom")
    .agg(
        samples=(
            "sample_token",
            "count",
        ),

        exact_matches=(
            "same_candidates",
            "sum",
        ),

        agreement_rate=(
            "same_candidates",
            "mean",
        ),

        minimum_jaccard=(
            "candidate_jaccard",
            "min",
        ),

        mean_jaccard=(
            "candidate_jaccard",
            "mean",
        ),

        fixed_only=(
            "fixed_only_count",
            "sum",
        ),

        hierarchical_only=(
            "hierarchical_only_count",
            "sum",
        ),
    )
)

agreement_4d

,samples,exact_matches,agreement_rate,minimum_jaccard,mean_jaccard,fixed_only,hierarchical_only
zoom,,,,,,,
23,404,404,1.0,1.0,1.0,0,0
24,404,404,1.0,1.0,1.0,0,0


In [71]:
mismatches_4d = (
    results_4d_df[
        ~results_4d_df[
            "same_candidates"
        ]
    ]
)

print(
    "Candidate mismatches:",
    len(mismatches_4d),
)

mismatches_4d[
    [
        "zoom",
        "scene",
        "location",

        "ego_yaw_deg",

        "fixed_cells",
        "hierarchical_cells",

        "fixed_candidates",
        "hierarchical_candidates",

        "candidate_jaccard",

        "fixed_only_count",
        "hierarchical_only_count",
    ]
].head(30)

Candidate mismatches: 0


,zoom,scene,location,ego_yaw_deg,fixed_cells,hierarchical_cells,fixed_candidates,hierarchical_candidates,candidate_jaccard,fixed_only_count,hierarchical_only_count


In [72]:
summary_4d = (
    results_4d_df
    .groupby("zoom")
    .agg(
        # ====================================================
        # Dataset
        # ====================================================

        samples=(
            "sample_token",
            "count",
        ),

        # ====================================================
        # Equivalence
        # ====================================================

        exact_candidate_matches=(
            "same_candidates",
            "sum",
        ),

        agreement_rate=(
            "same_candidates",
            "mean",
        ),

        min_candidate_jaccard=(
            "candidate_jaccard",
            "min",
        ),

        # ====================================================
        # Representation
        # ====================================================

        mean_fixed_cells=(
            "fixed_cells",
            "mean",
        ),

        median_fixed_cells=(
            "fixed_cells",
            "median",
        ),

        p95_fixed_cells=(
            "fixed_cells",
            lambda x:
                x.quantile(0.95),
        ),

        mean_hierarchical_cells=(
            "hierarchical_cells",
            "mean",
        ),

        median_hierarchical_cells=(
            "hierarchical_cells",
            "median",
        ),

        p95_hierarchical_cells=(
            "hierarchical_cells",
            lambda x:
                x.quantile(0.95),
        ),

        mean_compression_ratio=(
            "compression_ratio",
            "mean",
        ),

        median_compression_ratio=(
            "compression_ratio",
            "median",
        ),

        mean_reduction_fraction=(
            "reduction_fraction",
            "mean",
        ),

        # ====================================================
        # Over-coverage
        # ====================================================

        mean_fixed_overcoverage=(
            "fixed_overcoverage_ratio",
            "mean",
        ),

        median_fixed_overcoverage=(
            "fixed_overcoverage_ratio",
            "median",
        ),

        mean_hierarchical_overcoverage=(
            "hierarchical_overcoverage_ratio",
            "mean",
        ),

        median_hierarchical_overcoverage=(
            "hierarchical_overcoverage_ratio",
            "median",
        ),

        p95_hierarchical_overcoverage=(
            "hierarchical_overcoverage_ratio",
            lambda x:
                x.quantile(0.95),
        ),

        # ====================================================
        # Retrieval
        # ====================================================

        fixed_tp=(
            "fixed_tp",
            "sum",
        ),

        fixed_fp=(
            "fixed_fp",
            "sum",
        ),

        fixed_fn=(
            "fixed_fn",
            "sum",
        ),

        hierarchical_tp=(
            "hierarchical_tp",
            "sum",
        ),

        hierarchical_fp=(
            "hierarchical_fp",
            "sum",
        ),

        hierarchical_fn=(
            "hierarchical_fn",
            "sum",
        ),

        # ====================================================
        # Cover latency
        # ====================================================

        median_fixed_cover_ms=(
            "fixed_cover_time_ms",
            "median",
        ),

        p95_fixed_cover_ms=(
            "fixed_cover_time_ms",
            lambda x:
                x.quantile(0.95),
        ),

        median_hierarchical_cover_ms=(
            "hierarchical_cover_time_ms",
            "median",
        ),

        p95_hierarchical_cover_ms=(
            "hierarchical_cover_time_ms",
            lambda x:
                x.quantile(0.95),
        ),

        # ====================================================
        # Lookup latency
        # ====================================================

        median_fixed_lookup_ms=(
            "fixed_lookup_time_ms",
            "median",
        ),

        p95_fixed_lookup_ms=(
            "fixed_lookup_time_ms",
            lambda x:
                x.quantile(0.95),
        ),

        median_hierarchical_lookup_ms=(
            "hierarchical_lookup_time_ms",
            "median",
        ),

        p95_hierarchical_lookup_ms=(
            "hierarchical_lookup_time_ms",
            lambda x:
                x.quantile(0.95),
        ),

        # ====================================================
        # Total latency
        # ====================================================

        mean_fixed_total_ms=(
            "fixed_total_time_ms",
            "mean",
        ),

        median_fixed_total_ms=(
            "fixed_total_time_ms",
            "median",
        ),

        p95_fixed_total_ms=(
            "fixed_total_time_ms",
            lambda x:
                x.quantile(0.95),
        ),

        mean_hierarchical_total_ms=(
            "hierarchical_total_time_ms",
            "mean",
        ),

        median_hierarchical_total_ms=(
            "hierarchical_total_time_ms",
            "median",
        ),

        p95_hierarchical_total_ms=(
            "hierarchical_total_time_ms",
            lambda x:
                x.quantile(0.95),
        ),

        # ====================================================
        # Speedup
        # ====================================================

        mean_total_speedup=(
            "total_speedup",
            "mean",
        ),

        median_total_speedup=(
            "total_speedup",
            "median",
        ),

        min_total_speedup=(
            "total_speedup",
            "min",
        ),

        p05_total_speedup=(
            "total_speedup",
            lambda x:
                x.quantile(0.05),
        ),

        median_lookup_speedup=(
            "lookup_speedup",
            "median",
        ),
    )
    .reset_index()
)

In [73]:
summary_4d[
    "fixed_precision"
] = (
    summary_4d[
        "fixed_tp"
    ]
    /
    (
        summary_4d[
            "fixed_tp"
        ]
        +
        summary_4d[
            "fixed_fp"
        ]
    )
)


summary_4d[
    "fixed_recall"
] = (
    summary_4d[
        "fixed_tp"
    ]
    /
    (
        summary_4d[
            "fixed_tp"
        ]
        +
        summary_4d[
            "fixed_fn"
        ]
    )
)


summary_4d[
    "hierarchical_precision"
] = (
    summary_4d[
        "hierarchical_tp"
    ]
    /
    (
        summary_4d[
            "hierarchical_tp"
        ]
        +
        summary_4d[
            "hierarchical_fp"
        ]
    )
)


summary_4d[
    "hierarchical_recall"
] = (
    summary_4d[
        "hierarchical_tp"
    ]
    /
    (
        summary_4d[
            "hierarchical_tp"
        ]
        +
        summary_4d[
            "hierarchical_fn"
        ]
    )
)

In [74]:
final_4d = (
    summary_4d[
        [
            "zoom",
            "samples",

            "agreement_rate",
            "min_candidate_jaccard",

            "fixed_precision",
            "fixed_recall",

            "hierarchical_precision",
            "hierarchical_recall",

            "mean_fixed_cells",
            "mean_hierarchical_cells",

            "mean_compression_ratio",
            "median_compression_ratio",

            "mean_hierarchical_overcoverage",
            "p95_hierarchical_overcoverage",

            "median_fixed_cover_ms",
            "median_hierarchical_cover_ms",

            "median_fixed_lookup_ms",
            "median_hierarchical_lookup_ms",

            "median_fixed_total_ms",
            "median_hierarchical_total_ms",

            "p95_fixed_total_ms",
            "p95_hierarchical_total_ms",

            "median_lookup_speedup",
            "median_total_speedup",

            "p05_total_speedup",
            "min_total_speedup",
        ]
    ]
)

final_4d

,zoom,samples,agreement_rate,min_candidate_jaccard,fixed_precision,fixed_recall,hierarchical_precision,hierarchical_recall,mean_fixed_cells,mean_hierarchical_cells,...,median_fixed_lookup_ms,median_hierarchical_lookup_ms,median_fixed_total_ms,median_hierarchical_total_ms,p95_fixed_total_ms,p95_hierarchical_total_ms,median_lookup_speedup,median_total_speedup,p05_total_speedup,min_total_speedup
0,23,404,1.0,1.0,0.909353,1.0,0.909353,1.0,249.245050,112.529703,...,0.550271,0.290271,9.730020,4.213375,18.126308,7.591249,2.058493,2.258787,1.848674,0.959582
1,24,404,1.0,1.0,0.944175,1.0,0.944175,1.0,919.089109,241.289604,...,1.979709,0.615980,36.715666,8.216312,68.547575,14.765652,3.497378,4.429280,3.622300,3.484869


In [75]:
final_4d.to_csv(
    STEP4D_SUMMARY_OUTPUT,
    index=False,
)

print(
    "Saved:",
    STEP4D_SUMMARY_OUTPUT,
)

Saved: /Users/abinavbala/College/Research/BST-BEV/outputs/step4d_oriented_summary.csv


In [76]:
display_4d = (
    final_4d.copy()
)

display_4d[
    "mean_hierarchical_overcoverage_pct"
] = (
    100
    * display_4d[
        "mean_hierarchical_overcoverage"
    ]
)

display_4d[
    "p95_hierarchical_overcoverage_pct"
] = (
    100
    * display_4d[
        "p95_hierarchical_overcoverage"
    ]
)

display_4d[
    "descriptor_reduction_pct"
] = (
    100
    * (
        1
        - display_4d[
            "mean_hierarchical_cells"
        ]
        / display_4d[
            "mean_fixed_cells"
        ]
    )
)

display_4d[
    [
        "zoom",

        "agreement_rate",

        "fixed_recall",
        "hierarchical_recall",

        "mean_fixed_cells",
        "mean_hierarchical_cells",

        "descriptor_reduction_pct",

        "mean_hierarchical_overcoverage_pct",

        "median_fixed_total_ms",
        "median_hierarchical_total_ms",

        "median_total_speedup",
    ]
]

,zoom,agreement_rate,fixed_recall,hierarchical_recall,mean_fixed_cells,mean_hierarchical_cells,descriptor_reduction_pct,mean_hierarchical_overcoverage_pct,median_fixed_total_ms,median_hierarchical_total_ms,median_total_speedup
0,23,1.0,1.0,1.0,249.245050,112.529703,54.851780,19.333478,9.730020,4.213375,2.258787
1,24,1.0,1.0,1.0,919.089109,241.289604,73.746876,9.536918,36.715666,8.216312,4.429280


In [77]:
def axis_alignment_angle(
    yaw_rad,
):
    """
    Angular distance to the nearest global X/Y axis.

    Result:
        0 deg  -> axis aligned
        45 deg -> maximally diagonal
    """

    angle = (
        abs(
            math.degrees(
                yaw_rad
            )
        )
        % 90.0
    )

    return min(
        angle,
        90.0 - angle,
    )


results_4d_df[
    "axis_alignment_deg"
] = (
    results_4d_df[
        "ego_yaw_rad"
    ].apply(
        axis_alignment_angle
    )
)

In [78]:
results_4d_df[
    "orientation_bin"
] = pd.cut(
    results_4d_df[
        "axis_alignment_deg"
    ],

    bins=[
        0,
        10,
        20,
        30,
        40,
        45.0001,
    ],

    include_lowest=True,
)

In [79]:
orientation_analysis = (
    results_4d_df
    .groupby(
        [
            "zoom",
            "orientation_bin",
        ],
        observed=True,
    )
    .agg(
        samples=(
            "sample_token",
            "count",
        ),

        mean_fixed_cells=(
            "fixed_cells",
            "mean",
        ),

        mean_hierarchical_cells=(
            "hierarchical_cells",
            "mean",
        ),

        median_compression=(
            "compression_ratio",
            "median",
        ),

        median_overcoverage=(
            "hierarchical_overcoverage_ratio",
            "median",
        ),

        median_total_ms=(
            "hierarchical_total_time_ms",
            "median",
        ),

        median_speedup=(
            "total_speedup",
            "median",
        ),
    )
    .reset_index()
)

orientation_analysis

,zoom,orientation_bin,samples,mean_fixed_cells,mean_hierarchical_cells,median_compression,median_overcoverage,median_total_ms,median_speedup
0,23,"(-0.001, 10.0]",56,287.535714,101.089286,2.902260,0.125543,5.309541,1.874699
1,23,"(10.0, 20.0]",38,212.026316,97.763158,2.092869,0.197578,3.753792,2.067856
2,23,"(20.0, 30.0]",96,246.989583,112.458333,2.042553,0.216572,4.172813,2.209896
3,23,"(30.0, 40.0]",92,232.858696,113.619565,1.935044,0.226085,4.173708,2.338872
4,23,"(40.0, 45.0]",122,257.393443,121.614754,1.921949,0.210249,4.298916,2.401859
5,24,"(-0.001, 10.0]",56,1081.517857,214.214286,5.250087,0.061474,10.447958,3.675688
6,24,"(10.0, 20.0]",38,780.657895,208.947368,3.601036,0.099364,7.214312,3.993616
7,24,"(20.0, 30.0]",96,909.895833,241.697917,3.393204,0.105690,8.122292,4.347875
8,24,"(30.0, 40.0]",92,851.413043,243.760870,3.222477,0.112029,8.135938,4.540628
9,24,"(40.0, 45.0]",122,945.918033,261.606557,3.237882,0.108869,8.425103,4.601440
